# E01 Matrix Sensing Benchmark — PyTorch

This notebook is the PyTorch notebook-first rewrite of the Matrix Sensing benchmark. It keeps the experiment visible in one place:

- target matrix generation
- measurement tensor generation
- matrix sensing loss
- optimizer selection
- run loop
- metrics
- plots
- conclusion

Official `torch.optim.Muon` is used when available; local exact-SVD Muon and Shampoo helpers come from `muonlib_torch`.

Default mode is the full E01 grid: 5 optimizers, 3 dimensions, 10 seeds, and 2000 iterations per run. Quick validation lives outside the notebook in `smoketests/`. Results stay in notebook memory as `df` and `trajectories`; this notebook does not write CSV, PNG, or report files.

In [ ]:
from pathlib import Path
import os
import sys
from concurrent.futures import ProcessPoolExecutor, as_completed
from multiprocessing import get_context

for thread_env in ["OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS", "NUMEXPR_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"]:
    os.environ.setdefault(thread_env, "1")

import matplotlib.pyplot as plt
import pandas as pd
import torch
from IPython.display import Markdown, display
from tqdm.auto import tqdm

PROJECT = Path.cwd().resolve()
if not (PROJECT / "muonlib_torch").exists():
    PROJECT = PROJECT.parent.resolve()
sys.path.insert(0, str(PROJECT))

from muonlib_torch.e01_matrix_sensing import run_spec as run_e01_spec

torch.set_default_dtype(torch.float64)
torch.set_num_threads(1)
try:
    torch.set_num_interop_threads(1)
except RuntimeError:
    pass

print(f"project = {PROJECT}")
print(f"torch   = {torch.__version__}")

## Parameters

These constants define the experiment. The notebook should make the run grid obvious without opening any library code.

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.float64

ALGOS = ["Muon", "Muon-Exact", "Shampoo", "Adam", "SGD"]
DIMS = [50, 60, 70]
RANK = 5
LR = 0.01
NOISE = 0.0
DIST = "normal"
SPECTRUM = "hard-cutoff"
KAPPA = 1.0
INIT_SCALE = 0.01
ITERS = 2000
SEEDS = list(range(10))
EPSILON = 0.01
NUM_WORKERS = min(8, os.cpu_count() or 1)

print(f"device={DEVICE}, dtype={DTYPE}")
print(f"workers={NUM_WORKERS}")
print(f"runs={len(ALGOS) * len(DIMS) * len(SEEDS)}, iters={ITERS}")

In [ ]:
def grid_preview():
    rows = []
    for algo in ALGOS:
        for d in DIMS:
            m_meas = 2 * d * RANK
            rows.append({
                "algo": algo,
                "d": d,
                "rank": RANK,
                "m_meas": m_meas,
                "seeds": len(SEEDS),
                "iters_per_seed": ITERS,
                "total_steps": len(SEEDS) * ITERS,
            })
    return pd.DataFrame(rows)

display(grid_preview())

## Matrix Sensing Problem

Target matrix:

$$
X^\star = U \operatorname{diag}(s)V^\top
$$

Measurements:

$$
y_i = \langle A_i, X^\star\rangle + \varepsilon_i
$$

Loss:

$$
f(X) = \frac{1}{2m}\sum_{i=1}^{m}(\langle A_i, X\rangle-y_i)^2
$$

The key PyTorch expression is `torch.einsum("mij,ij->m", A, X)`, which computes all matrix inner products at once.

The per-run implementation lives in `muonlib_torch.e01_matrix_sensing.run_spec` and is imported as `run_e01_spec`. This keeps each worker function importable for `ProcessPoolExecutor` with the `spawn` start method while the experiment grid stays visible in this notebook.

## Optimizers

`Muon` uses official `torch.optim.Muon` when available and falls back to `MuonTorch` on older PyTorch; `Muon-Exact` uses local exact-SVD `MuonTorch`; `Shampoo` uses `ShampooTorch`; `Adam` and `SGD` use built-in PyTorch optimizers. The optimizer choice is intentionally visible here.

Optimizer construction is part of the importable worker. The compared methods are the names listed in `ALGOS` in the parameter cell.

## Single Run Worker

Each worker returns one result row and one trajectory. `K_epsilon` is recorded when the loss first crosses the threshold, but the loop does not break early, so all runs keep the same iteration budget for plotting.

The worker receives a plain dictionary containing `algo`, `d`, `seed`, optimizer parameters, and numerical settings. That dictionary is what the process pool serializes.

## Metrics and Plots

The result table is one row per run. The trajectory dictionary stores per-step loss and gradient norm for plotting inside the notebook.

In [ ]:
def summary_table(df):
    return df.groupby(["algo", "d"], as_index=False).agg(
        runs=("seed", "count"),
        K_epsilon_mean=("K_epsilon", "mean"),
        K_epsilon_std=("K_epsilon", "std"),
        min_loss_mean=("min_loss", "mean"),
        final_loss_mean=("final_loss", "mean"),
        time_s_mean=("time_s", "mean"),
    )


def trajectory_frame(trajectories):
    records = []
    for (algo, d, seed), traj in trajectories.items():
        for step, loss in enumerate(traj["loss"]):
            records.append({"algo": algo, "d": d, "seed": seed, "step": step, "loss": loss})
    return pd.DataFrame(records)


def plot_loss_curves(trajectories):
    if not trajectories:
        return
    tf = trajectory_frame(trajectories)
    mean_tf = tf.groupby(["algo", "d", "step"], as_index=False)["loss"].mean()

    plt.figure(figsize=(8.5, 4.8))
    for (algo, d, seed), traj in list(trajectories.items())[:12]:
        plt.plot(traj["loss"], color="0.75", linewidth=1, alpha=0.45)
    for (algo, d), sub in mean_tf.groupby(["algo", "d"]):
        plt.plot(sub["step"], sub["loss"], linewidth=2.4, label=f"{algo} d={d} mean")
    plt.yscale("log")
    plt.xlabel("step")
    plt.ylabel("loss")
    plt.title("Matrix sensing loss trajectories")
    plt.legend(fontsize=8)
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()


def plot_summary(df):
    if df.empty:
        return
    summary = summary_table(df)
    display(df.sort_values(["algo", "d", "seed"]).reset_index(drop=True))
    display(summary)

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    for algo, sub in summary.groupby("algo"):
        axes[0].plot(sub["d"], sub["K_epsilon_mean"], marker="o", label=algo)
        axes[1].plot(sub["d"], sub["time_s_mean"], marker="o", label=algo)
    axes[0].set_title("Mean K_epsilon")
    axes[0].set_xlabel("d")
    axes[0].set_ylabel("iterations")
    axes[1].set_title("Mean wall-clock")
    axes[1].set_xlabel("d")
    axes[1].set_ylabel("seconds")
    for ax in axes:
        ax.grid(alpha=0.3)
        ax.legend()
    plt.tight_layout()
    plt.show()

## Run the Experiment

Each `(algo, d, seed)` tuple is an independent run. `run_experiment()` dispatches those runs across worker processes and uses `tqdm` to show completed runs.

In [ ]:
def build_run_specs():
    specs = []
    for algo in ALGOS:
        for d in DIMS:
            for seed in SEEDS:
                specs.append({
                    "algo": algo,
                    "d": d,
                    "rank": RANK,
                    "lr": LR,
                    "noise": NOISE,
                    "dist": DIST,
                    "spectrum": SPECTRUM,
                    "kappa": KAPPA,
                    "init_scale": INIT_SCALE,
                    "seed": seed,
                    "iters": ITERS,
                    "epsilon": EPSILON,
                    "device_type": DEVICE.type,
                    "dtype_name": "float64",
                })
    return specs


def sort_results(df):
    if df.empty:
        return df
    ordered = df.copy()
    ordered["algo"] = pd.Categorical(ordered["algo"], categories=ALGOS, ordered=True)
    ordered = ordered.sort_values(["algo", "d", "seed"]).reset_index(drop=True)
    ordered["algo"] = ordered["algo"].astype(str)
    return ordered


def run_experiment(num_workers=NUM_WORKERS):
    specs = build_run_specs()
    num_workers = max(1, min(int(num_workers), len(specs)))
    rows = []
    trajectories = {}

    if num_workers == 1:
        for spec in tqdm(specs, desc="E01 runs", unit="run"):
            key, row, traj = run_e01_spec(spec)
            rows.append(row)
            trajectories[key] = traj
        return sort_results(pd.DataFrame(rows)), trajectories

    with ProcessPoolExecutor(max_workers=num_workers, mp_context=get_context("spawn")) as executor:
        futures = [executor.submit(run_e01_spec, spec) for spec in specs]
        for future in tqdm(as_completed(futures), total=len(futures), desc=f"E01 runs ({num_workers} proc)", unit="run"):
            key, row, traj = future.result()
            rows.append(row)
            trajectories[key] = traj

    return sort_results(pd.DataFrame(rows)), trajectories

## Execute

Run the next cell to start the experiment. It leaves results in memory only:

- `df`: one row per run
- `trajectories`: per-step loss and gradient norm

In [ ]:
df, trajectories = run_experiment()
plot_summary(df)
plot_loss_curves(trajectories)

## Conclusion

This cell derives a short conclusion from the in-memory result table.

In [ ]:
def conclusion_markdown(df):
    summary = summary_table(df)
    lines = []
    lines.append("### Result Summary")
    lines.append("")
    lines.append(f"- Runs: `{len(df)}`")
    lines.append(f"- Methods: `{', '.join(sorted(df['algo'].unique()))}`")
    lines.append(f"- Iterations per run: `{int(df['iters'].iloc[0])}`")
    lines.append(f"- Total optimizer steps: `{len(df) * int(df['iters'].iloc[0])}`")
    lines.append("")

    for d in sorted(df["d"].unique()):
        sub = summary[summary["d"] == d]
        best_k = sub.loc[sub["K_epsilon_mean"].idxmin()]
        best_time = sub.loc[sub["time_s_mean"].idxmin()]
        best_loss = sub.loc[sub["min_loss_mean"].idxmin()]
        lines.append(
            f"- d={d}: best K_epsilon is `{best_k.algo}` at `{best_k.K_epsilon_mean:.1f}` steps; "
            f"fastest wall-clock is `{best_time.algo}` at `{best_time.time_s_mean:.3f}s`; "
            f"lowest min_loss is `{best_loss.algo}` at `{best_loss.min_loss_mean:.3e}`."
        )

    if {"Muon", "Muon-Exact"}.issubset(set(df["algo"])):
        lines.append("")
        lines.append("Muon vs Muon-Exact checks whether the Newton-Schulz approximation tracks the exact SVD polar direction.")
    if "Shampoo" in set(df["algo"]):
        lines.append("Shampoo is included as a second-order preconditioned baseline; compare both K_epsilon and wall-clock, because its eigendecompositions are not free.")
    return "\n".join(lines)


display(Markdown(conclusion_markdown(df)))